# 02 — AUC analysis

This notebook never fits a model. It reads saved continuous scores from `outputs/pysr/`, computes ROC-AUC and related metrics, and writes analysis artifacts under `outputs/auc/`.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    precision_score, recall_score, roc_auc_score, roc_curve,
)

def repository_root() -> Path:
    candidate = Path.cwd().resolve()
    for _ in range(6):
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
        candidate = candidate.parent
    raise FileNotFoundError('Could not locate repository root from notebook directory.')

ROOT = repository_root()
RUN_ID = 'notebook_pysr_run_v1'
SCORES_PATH = ROOT / 'outputs' / 'pysr' / RUN_ID / 'scores.csv'
ANALYSIS_DIR = ROOT / 'outputs' / 'auc' / RUN_ID
THRESHOLD = 0.5
print(f'Scores: {SCORES_PATH}')
print(f'Analysis output: {ANALYSIS_DIR}')

In [ ]:
if not SCORES_PATH.is_file():
    raise FileNotFoundError(f'Missing saved scores: {SCORES_PATH}. Run 01_pysr_run.ipynb with an authorized fit first.')
scores = pd.read_csv(SCORES_PATH)
required = {'row_index', 'split', 'y_true', 'score', 'score_source'}
missing = sorted(required - set(scores.columns))
if missing:
    raise ValueError(f'Scores file is missing columns: {missing}')
if set(scores['split'].dropna().unique()) != {'test'}:
    raise ValueError('AUC analysis requires test-only saved scores.')
y_true = scores['y_true'].to_numpy()
continuous_scores = scores['score'].to_numpy(dtype=float)
if set(np.unique(y_true)) != {0, 1}:
    raise ValueError('AUC analysis requires binary labels 0 and 1.')
if not np.isfinite(continuous_scores).all() or len(np.unique(continuous_scores)) < 2:
    raise ValueError('Scores must be finite and non-constant continuous values.')
print(f"Loaded {len(scores)} continuous test scores from {scores['score_source'].iloc[0]!r}.")

In [ ]:
roc_auc = float(roc_auc_score(y_true, continuous_scores))
average_precision = float(average_precision_score(y_true, continuous_scores))
fpr, tpr, thresholds = roc_curve(y_true, continuous_scores)
hard_predictions = (continuous_scores >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, hard_predictions, labels=[0, 1]).ravel()
metrics = {
    'run_id': RUN_ID,
    'score_source': str(scores['score_source'].iloc[0]),
    'auc_rule': 'continuous_scores_only',
    'roc_auc': roc_auc,
    'average_precision': average_precision,
    'test_rows': int(len(scores)),
    'threshold': THRESHOLD,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'review_status': 'provisional, unverified, pending review',
}
threshold_metrics = pd.DataFrame([{
    'threshold': THRESHOLD,
    'accuracy': accuracy_score(y_true, hard_predictions),
    'precision': precision_score(y_true, hard_predictions, zero_division=0),
    'recall': recall_score(y_true, hard_predictions, zero_division=0),
    'true_negative': int(tn), 'false_positive': int(fp),
    'false_negative': int(fn), 'true_positive': int(tp),
}])
display(pd.DataFrame([metrics]))
display(threshold_metrics)

In [ ]:
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
(ANALYSIS_DIR / 'figures').mkdir(exist_ok=True)
(ANALYSIS_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'threshold': thresholds}).to_csv(ANALYSIS_DIR / 'roc_curve.csv', index=False)
threshold_metrics.to_csv(ANALYSIS_DIR / 'threshold_metrics.csv', index=False)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.4f}')
ax.plot([0, 1], [0, 1], '--', color='0.5', label='Chance')
ax.set(xlabel='False positive rate', ylabel='True positive rate', title=f'Continuous-score ROC — {RUN_ID}')
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / 'figures' / 'roc_curve.png', dpi=160)
plt.show()
print(f'Wrote AUC analysis to {ANALYSIS_DIR}')